# Bird-detection results

Cross-model comparison of the detectors over all (backend × threshold-objective) cells, then macro metric-vs-threshold curves for every model. Rendered from saved `results.json` / `.npz`; no retraining.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path
import os
import pyrootutils

os.environ.setdefault('TF_CPP_MIN_LOG_LEVEL', '3')

ROOT = pyrootutils.setup_root(
    search_from=Path.cwd(),
    indicator='pyproject.toml',
    pythonpath=True,
    dotenv=True,
)

## Parameters

In [ ]:
from building.polychirp_training import detection_slug
from building.models import BASE_LIST

LAT, LON = 48.8566, 2.3522
MODELS = BASE_LIST  # only these (build_model, multiplier) pairs are shown

COLLECTION = detection_slug(LAT, LON)
TASK_DIR = ROOT / "models" / COLLECTION
print(f"task: {COLLECTION}   dir: {TASK_DIR}  exists={TASK_DIR.exists()}")


## Cross-model comparison

Macro metrics on the test split, four tables: (INT8 | float) × (F1-opt | F2-opt threshold).

In [ ]:
from building.polychirp_training.results import gather_macro_table

keep = {(m.name, m.multi) for m in MODELS}
collapse = {m.name for m in MODELS if m.base_model.quantize_collapse}
metric_cols = ["macro_precision", "macro_recall", "macro_f1", "macro_f2", "macro_auc"]

def comparison(backend, objective):
    df = gather_macro_table(TASK_DIR, backend, objective)
    df = df[[(b, m) in keep for b, m in zip(df["build_model"], df["multiplier"])]]
    if backend == "tflite":  # INT8 collapse models skew the gradient scale; drop them
        df = df[~df["build_model"].isin(collapse)]
    return (
        df[metric_cols + ["flash_kb"]].style
        .background_gradient(cmap="Blues", subset=metric_cols)
        .format("{:.3f}", subset=metric_cols, na_rep="-")
        .format({"flash_kb": "{:.1f}"}, na_rep="-")
        .set_caption(f"{'INT8' if backend == 'tflite' else 'float'} — {objective.upper()}-tuned threshold  ({COLLECTION})")
    )

for backend in ("tflite", "keras"):
    for objective in ("f1", "f2"):
        display(comparison(backend, objective))


## Metric vs threshold (all models)

INT8, macro-averaged over target classes, one curve per model. 2×3 panels.

In [ ]:
import matplotlib.pyplot as plt

from building.polychirp_training.results import load_run, run_model_label
from building.models import model_eval as M

evals = {}
for m in MODELS:
    if m.base_model.quantize_collapse:  # INT8 collapses; keep it off the quantized curves
        continue
    paths = m.paths(COLLECTION)
    if paths.results_path.exists():
        evals[run_model_label(m.name, m.multi)] = load_run(paths.results_path).eval("tflite", "test", "f2")

M.plot_model_sweep_panels(evals); plt.show()


## Export for the paper

Freeze to `<task>/final_results/`: `detection.json` (per-model macro metrics + float/INT8 loss), the float-vs-INT8 loss-drop figure (CSV+PDF, leaf collapse visible), and a macro metric-vs-threshold sweep CSV+PDF per metric. Zip that folder for the paper machine.

In [ ]:
from building.polychirp_training.export import export_detection

# collapse comes from the cross-model comparison cell above.
export_detection(TASK_DIR, COLLECTION, MODELS, collapse)
